# Postprocessing SPH Particle Tracks with UCLCHEM

Many astrochemical models assume simple, idealised physical conditions — a uniform-density
cloud or a standard free-fall collapse. In reality, the gas parcels that eventually form
stars follow complex trajectories through a turbulent, self-gravitating interstellar medium.
Simulations such as the **NEATH** (Non-Equilibrium Abundances Treated Holistically)
project record these trajectories as SPH particle tracks: time series of density,
temperature, column densities, and positions for many individual gas parcels.

UCLCHEM's `Postprocess` class lets you drive the chemical network with *any* time-varying
physical structure you can describe as a set of arrays or a function. The framework is based
on NEATH's output format, but it is flexible enough to serve many purposes, even
without column densities and other hydrodynamical quantities.

This notebook is designed to be **run and interpreted** rather than filled in. All code is
provided. As you work through it, focus on the questions in the text — they are the real
exercises.

We will:

1. Download a small NEATH example dataset.
2. Load the SPH particle tracks and explore their physical diversity.
3. Run `Postprocess` on a single particle and inspect the chemistry.
4. Run a small ensemble of particles and compare their chemical outcomes.
5. Add a free-fall collapse as a reference model.
6. Design and run your own physical history from scratch.

## 1  Setup

We start by importing everything we need. `uclchem` is the main package;
`urllib.request` handles the data download without any extra dependency.

We also set the OpenMP and BLAS thread counts to 1 **before** importing numpy or uclchem.
This is mostly a preventative measure for Apple machines, where the Fortran backend can
otherwise pin all threads to a single core.

In [ ]:
import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import uclchem

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

# Species we ask UCLCHEM to track in every run
OUT_SPECIES = [
    "H+",
    "H2",
    "H",
    "CO",
    "HCO+",
    "CH3OH",
    "H2O",
    "CH4",
    "NH3",
    "H3O+",
    "C+",
    "O",
    "C",
]

## 2  Downloading the NEATH Example Data

The NEATH project provides SPH particle tracks from a self-gravitating turbulent cloud
simulation. We fetch a small example file from the public NEATH GitHub repository.
The cell checks whether the file already exists so that re-running the notebook does not
trigger an unnecessary download.

In [ ]:
NEATH_URL = (
    "https://raw.githubusercontent.com/fpriestley/neath/"
    "8dc6a1f49a3dba8d0ea7ae528d4f2753975a8c47/data/neath_example_data.out"
)
NEATH_FILE = DATA_DIR / "neath_example_data_250430.out"

if NEATH_FILE.exists():
    print(f"{NEATH_FILE.name} already present, skipping download.")
else:
    print(f"Downloading {NEATH_FILE.name} ...")
    urllib.request.urlretrieve(NEATH_URL, NEATH_FILE)
    print("Done.")

## 3  Loading and Exploring the SPH Particle Tracks

The NEATH output file is a plain-text, whitespace-separated table. Each row corresponds
to one particle at one timestep. A new particle begins whenever the time column resets to
zero. The 16 columns are:

| # | Name | Description |
|---|------|-------------|
| 0 | `time` | Time in seconds |
| 1–3 | `x y z` | Position (not used here) |
| 4 | `density` | Number density $n_\mathrm{H}$ / cm$^{-3}$ |
| 5–7 | `vx vy vz` | Velocity (not used here) |
| 8 | `Tgas` | Gas temperature / K |
| 9 | `xH2` | H$_2$ abundance fraction (not passed to UCLCHEM) |
| 10 | `xCO` | CO abundance fraction (not passed to UCLCHEM) |
| 11 | `col12` | Unused |
| 12 | `N_H` | Hydrogen column density / cm$^{-2}$ |
| 13 | `N_H2` | H$_2$ column density / cm$^{-2}$ |
| 14 | `N_CO` | CO column density / cm$^{-2}$ |
| 15 | `Tdust` | Dust temperature / K |

UCLCHEM expects time in *years*, so we convert on loading.

In [ ]:
SECONDS_PER_YEAR = 24 * 3600 * 365.25

NEATH_COLUMNS = [
    "time",
    "x",
    "y",
    "z",
    "density",
    "vx",
    "vy",
    "vz",
    "Tgas",
    "xH2",
    "xCO",
    "col12",
    "N_H",
    "N_H2",
    "N_CO",
    "Tdust",
]


def load_neath_data(filepath) -> pd.DataFrame:
    """Load NEATH SPH data, assign column names, and label particles."""
    df = pd.read_csv(filepath, sep=r"\s+", header=None, dtype=np.float64)
    df.columns = NEATH_COLUMNS
    # A new particle starts whenever time resets to 0
    df["particle_id"] = (df["time"] == 0.0).astype(int).cumsum()
    df["time"] = df["time"] / SECONDS_PER_YEAR  # convert seconds → years
    return df


neath_df = load_neath_data(NEATH_FILE)
particle_ids = neath_df["particle_id"].unique()
n_timesteps = len(neath_df) // len(particle_ids)
print(f"Loaded {len(particle_ids)} particles with {n_timesteps} timesteps each.")
neath_df.head()

Before running any chemistry, it pays to look at the *physical* diversity of the
particle tracks. The function below overlays density, gas temperature, and dust
temperature for a selection of particles.

In [ ]:
def plot_particle_tracks(neath_df, ids=None, n_random=10, seed=42):
    """Overlay physical tracks for a selection of particles.

    Parameters
    ----------
    ids : list of int, optional
        Particle IDs to plot. If None, ``n_random`` IDs are chosen at random.
    n_random : int
        Number of random particles to pick when ``ids`` is None.
    """
    rng = np.random.default_rng(seed)
    all_ids = neath_df["particle_id"].unique()
    ids = (
        ids
        if ids is not None
        else rng.choice(all_ids, size=min(n_random, len(all_ids)), replace=False)
    )

    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
    configs = [
        ("density", r"$n_\mathrm{H}$ (cm$^{-3}$)", True),
        ("Tgas", "Gas temperature (K)", False),
        ("Tdust", "Dust temperature (K)", False),
    ]
    for ax, (col, ylabel, log) in zip(axes, configs):
        for pid in ids:
            p = neath_df[neath_df["particle_id"] == pid]
            t = p["time"] / 1e6
            (ax.semilogy if log else ax.plot)(t, p[col], linewidth=0.6, alpha=0.5)
        ax.set_xlabel("Time (Myr)")
        ax.set_ylabel(ylabel)

    plt.tight_layout()
    return fig, axes


fig, axes = plot_particle_tracks(neath_df)
plt.show()

> **Questions to consider:**
> - How much do the density histories differ between particles? What does that tell you
>   about the turbulent structure of the cloud?
> - Gas and dust temperatures are often assumed to be coupled. Is that a good assumption
>   for these particles?
> - Can you identify a particle that you would expect to have particularly rich ice
>   chemistry? What physical properties would you look for?

### Challenge: Select Your Particles

Rather than taking the first 10 particles, write a function `select_particles(neath_df, n=10)`
that returns `n` particle IDs based on a physical criterion you think will identify the most
chemically interesting particles. Some ideas:

- Particles that reach the **highest peak density**.
- Particles with the **largest temperature range** ($T_\mathrm{gas,max} - T_\mathrm{gas,min}$).
- Particles that spend the **most time above** a density threshold, e.g. $10^4\,\mathrm{cm}^{-3}$.

There is no single right answer — choose a criterion, implement it, and be ready to justify it.

In [ ]:
def select_particles(neath_df, n=10):
    """Return n particle IDs selected by a physical criterion of your choice."""
    # Your code here

In [ ]:
# Provided: call your function and visualise the selected tracks.
demo_ids = select_particles(neath_df, n=10)
print(f"Selected particle IDs: {list(demo_ids)}")
fig, axes = plot_particle_tracks(neath_df, ids=demo_ids)
plt.show()

## 4  Running Postprocessing on a Single Particle

`uclchem.model.Postprocess` accepts arrays for every physical quantity that varies with
time. Quantities that are effectively constant for these particles — here the
cosmic-ray ionisation rate $\zeta$ and UV field $F_{UV}$ — can be supplied as scalars.
The column density arrays ($N_\mathrm{H}$, $N_\mathrm{H_2}$, $N_\mathrm{CO}$) are used
by UCLCHEM to compute photodissociation rates self-consistently.

We save the result to an HDF5 file with `model.save_model()` so we can reload it later
without re-running the chemistry. The `uclchem.model.load_model()` function works for
all UCLCHEM model classes.

In [ ]:
def run_neath_particle(particle_df, model_file):
    """Run postprocessing on one NEATH particle; return the saved model file path."""
    if os.path.exists(model_file):
        print(f"  cached: {model_file}")
        return model_file
    model = uclchem.model.Postprocess(
        param_dict={"writeTimestepInfo": True},
        out_species=OUT_SPECIES,
        time_array=particle_df["time"].values,
        density_array=particle_df["density"].values,
        gas_temperature_array=particle_df["Tgas"].values,
        dust_temperature_array=particle_df["Tdust"].values,
        zeta_array=1.0,  # standard cosmic-ray ionisation rate
        radfield_array=1.0,  # 1 Habing unit UV field
        coldens_H_array=particle_df["N_H"].values,
        coldens_H2_array=particle_df["N_H2"].values,
        coldens_CO_array=particle_df["N_CO"].values,
        coldens_C_array=0.0,
    )
    model.save_model(file=model_file)
    print(f"  done: {model_file}")
    return model_file


p1_file = DATA_DIR / "neath_p1.h5"
run_neath_particle(neath_df[neath_df["particle_id"] == 1], p1_file)

p1_model = uclchem.model.load_model(file=p1_file)
physics_df, chemistry_df = p1_model.get_dataframes(joined=False)

The helper below plots the full abundance history for a set of species, showing gas
(solid), grain surface (dashed), and bulk ice (dotted) phases together. A faint density
profile is shown in the background for reference.

In [ ]:
def plot_species_evolution(physics_df, chemistry_df, species_list, ncols=3):
    """Plot gas, surface, and bulk abundances for each species in species_list.

    A faint density profile is overlaid on each panel for physical context.
    """
    nrows = (len(species_list) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False
    )
    time_myr = physics_df["Time"] / 1e6
    density = physics_df["Density"]

    phase_styles = [
        ("", "-", "gas"),
        ("#", "--", "surface"),
        ("@", ":", "bulk"),
    ]

    for i, sp in enumerate(species_list):
        ax = axes[i // ncols][i % ncols]

        # faint density background
        ax_twin = ax.twinx()
        ax_twin.semilogy(time_myr, density, color="0.75", linewidth=0.8, zorder=0)
        ax_twin.set_ylabel(r"$n_\mathrm{H}$", color="0.75", fontsize=7)
        ax_twin.tick_params(axis="y", labelcolor="0.75", labelsize=6)

        plotted = False
        for prefix, ls, label in phase_styles:
            col = f"{prefix}{sp}"
            if col in chemistry_df.columns:
                ax.semilogy(
                    time_myr,
                    chemistry_df[col],
                    linestyle=ls,
                    linewidth=1.4,
                    label=label,
                    zorder=2,
                )
                plotted = True
        if plotted:
            ax.legend(fontsize=7, loc="lower right")
        ax.set_title(sp, fontsize=9)
        ax.set_xlabel("Time (Myr)")
        ax.set_ylabel(r"Abundance $x_i$")

    for j in range(len(species_list), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    plt.tight_layout()
    return fig, axes


fig, axes = plot_species_evolution(
    physics_df, chemistry_df, ["H", "H2", "CO", "HCO+", "CH3OH", "H2O"]
)
plt.show()

> **Questions to consider:**
> - At what density does CO start to freeze out onto grain surfaces? Does this match the
>   canonical freeze-out density of $\sim 10^4\,\mathrm{cm}^{-3}$?
> - How does the gas-phase CO abundance respond when the density drops again?
> - Methanol is thought to form primarily on grain surfaces and then desorb. Can you see
>   this sequence in the plot? What triggers the desorption?
> - Why might HCO$^+$ behave differently from CO despite being chemically related?

## 5  Running a Small Ensemble of Particles

Running the same chemistry for multiple particles lets us see how sensitive the
chemical outcome is to the physical history. We process up to 10 particles here;
for the full dataset see `run.py` in this directory.

In [ ]:
demo_files = []
for pid in demo_ids[:2]:
    model_file = DATA_DIR / f"neath_p{pid}.h5"
    model = run_neath_particle(neath_df[neath_df["particle_id"] == pid], model_file)
    demo_files.append(model)

print(f"Processed {len(demo_files)} particles.")

The function below shows the spread across the ensemble. For each species it plots every
particle's abundance history as a thin line and shades the min–max envelope.

In [ ]:
def plot_abundance_spread(demo_files, species_list, ncols=3, reference=None):
    """Plot ensemble spread of gas-phase abundances across all demo models.

    Parameters
    ----------
    reference : tuple (physics_df, chemistry_df), optional
        A single reference model (e.g. free-fall) drawn as a thick dashed line.
    """
    nrows = (len(species_list) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(4 * ncols, 3 * nrows), squeeze=False
    )

    # collect all time series per species
    tracks = {sp: [] for sp in species_list}
    times = []
    for f in demo_files:
        m = uclchem.model.load_model(file=f)
        phys, chem = m.get_dataframes(joined=False)
        times.append(phys["Time"].values / 1e6)
        for sp in species_list:
            tracks[sp].append(chem[sp].values if sp in chem.columns else None)

    for i, sp in enumerate(species_list):
        ax = axes[i // ncols][i % ncols]
        for t, vals in zip(times, tracks[sp]):
            if vals is not None:
                ax.semilogy(t, vals, color="steelblue", linewidth=0.5, alpha=0.5)

        # min-max envelope
        valid = [(t, v) for t, v in zip(times, tracks[sp]) if v is not None]
        if valid:
            # interpolate onto a common time grid using the first particle's grid
            t_common = valid[0][0]
            stack = np.array([np.interp(t_common, t, v) for t, v in valid])
            ax.fill_between(
                t_common,
                stack.min(axis=0),
                stack.max(axis=0),
                color="steelblue",
                alpha=0.15,
            )

        if reference is not None:
            ref_phys, ref_chem = reference
            if sp in ref_chem.columns:
                ax.semilogy(
                    ref_phys["Time"] / 1e6,
                    ref_chem[sp],
                    color="tomato",
                    linewidth=1.8,
                    linestyle="--",
                    label="Free-fall",
                    zorder=5,
                )
                ax.legend(fontsize=7)

        ax.set_title(sp, fontsize=9)
        ax.set_xlabel("Time (Myr)")
        ax.set_ylabel(r"Abundance $x_i$")

    for j in range(len(species_list), nrows * ncols):
        axes[j // ncols][j % ncols].axis("off")

    plt.tight_layout()
    return fig, axes


fig, axes = plot_abundance_spread(
    demo_files, ["H2", "CO", "HCO+", "CH3OH", "H2O", "NH3"]
)
plt.show()

> **Questions to consider:**
> - Which species show the largest spread across particles? Which are most uniform?
> - What physical property of a particle do you think is most responsible for the
>   spread in methanol abundance?
> - H$_2$O and NH$_3$ both form primarily in the ice phase. Do they show similar or
>   different sensitivity to the physical history?

## 6  Free-Fall Collapse Reference Model

To put the NEATH results in context, we run a simple free-fall collapse from
$n = 10^2\,\mathrm{cm}^{-3}$ to $10^6\,\mathrm{cm}^{-3}$ using `uclchem.model.Cloud`.
This is the classic *phase 1* model in the UCLCHEM literature: it computes the density
evolution internally via the free-fall equation and produces a chemically evolved state
that can serve as a starting point for further models — or, as here, a simple reference.

Unlike `Postprocess`, `Cloud` does not require physical arrays; you only need to set the
initial and final conditions in the parameter dictionary.

In [ ]:
FF_FILE = DATA_DIR / "freefall_collapse.h5"

if not FF_FILE.exists():
    ff_model = uclchem.model.Cloud(
        param_dict={
            "initialDens": 1e2,  # starting density / cm^-3
            "finalDens": 1e6,  # final density / cm^-3
            "initialTemp": 10.0,  # isothermal temperature / K
            "freefall": True,  # use free-fall density evolution
            "finalTime": 6e6,  # maximum time / yr
        },
        out_species=OUT_SPECIES,
    )
    ff_model.save_model(file=FF_FILE)
    print(f"Saved {FF_FILE}")
else:
    print(f"Cached: {FF_FILE}")

ff_model = uclchem.model.load_model(file=FF_FILE)
ff_phys, ff_chem = ff_model.get_dataframes(joined=False)

Now overlay the free-fall reference on the ensemble spread.

In [ ]:
fig, axes = plot_abundance_spread(
    demo_files,
    ["H2", "CO", "HCO+", "CH3OH", "H2O", "NH3"],
    reference=(ff_phys, ff_chem),
)
plt.show()

> **Questions to consider:**
> - Does the free-fall reference lie within the NEATH ensemble spread, or outside it?
>   What does that tell you about how representative a simple collapse model is?
> - For which species does the free-fall model agree well with the NEATH ensemble?
>   For which does it diverge most?
> - The free-fall model uses a single, smooth density profile. What aspects of the
>   NEATH tracks might drive the chemical differences you observe?

## 7  Designing Your Own Physical History

One of the most powerful features of `Postprocess` is that it accepts *any* arrays you
provide. Below is a template you can modify to explore how individual physical parameters
affect the chemistry. Each section is labelled — try changing one parameter at a time
and re-running `plot_species_evolution` to build intuition.

In [ ]:
# ── Physical history template ──────────────────────────────────────────────────
def create_synthetic_data(n_samples=2000):
    """Return a dict of time-varying physical parameters spanning 10 Myr."""
    time = ...
    density = ...
    gas_temp = ...
    dust_temp = ...
    zeta = ...
    radfield = ...

    return dict(
        time=time,
        density=density,
        gas_temperature=gas_temp,
        dust_temperature=dust_temp,
        zeta=zeta,
        radfield=radfield,
    )


synth = create_synthetic_data()

# ──────────────────────────────────────────────────────────────────────────────

# Visualise the physical history before running the chemistry
fig, axes = plt.subplots(2, 3, figsize=(11, 5))
t_myr = synth["time"] / 1e6

axes[0, 0].semilogy(t_myr, synth["density"])
axes[0, 0].set(ylabel=r"$n_\mathrm{H}$ (cm$^{-3}$)", title="Density")
axes[0, 1].plot(t_myr, synth["gas_temperature"])
axes[0, 1].set(ylabel="Temperature (K)", title="Gas temperature")
axes[0, 2].plot(t_myr, synth["dust_temperature"])
axes[0, 2].set(ylabel="Temperature (K)", title="Dust temperature")
axes[1, 0].semilogy(t_myr, synth["zeta"])
axes[1, 0].set(ylabel=r"$\zeta$ (× standard)", title="CRIR")
axes[1, 1].plot(t_myr, synth["radfield"])
axes[1, 1].set(ylabel="$G_0$ (Habing)", title="UV field")
axes[1, 2].axis("off")

for ax in axes.flat:
    if ax.get_visible() and ax.lines:
        ax.set_xlabel("Time (Myr)")

plt.tight_layout()
plt.show()

In [ ]:
SYNTH_FILE = DATA_DIR / "postprocess_synthetic.h5"

# Delete the cached file if you change the physical history above, so the model re-runs.
if not SYNTH_FILE.exists():
    synth_model = uclchem.model.Postprocess(
        param_dict={"writeTimestepInfo": True},
        out_species=OUT_SPECIES,
        time_array=synth["time"],
        density_array=synth["density"],
        gas_temperature_array=synth["gas_temperature"],
        dust_temperature_array=synth["dust_temperature"],
        zeta_array=synth["zeta"],
        radfield_array=synth["radfield"],
        coldens_H_array=None,  # no column densities for a synthetic run
        coldens_H2_array=None,
        coldens_CO_array=None,
        coldens_C_array=None,
    )
    synth_model.save_model(file=SYNTH_FILE)
    print(f"Saved {SYNTH_FILE}")
else:
    print(
        f"Cached: {SYNTH_FILE}  (delete this file to re-run after changing parameters)"
    )

In [ ]:
synth_model = uclchem.model.load_model(file=SYNTH_FILE)
synth_phys, synth_chem = synth_model.get_dataframes(joined=False)

fig, axes = plot_species_evolution(
    synth_phys, synth_chem, ["H", "H2", "CO", "HCO+", "CH3OH", "H2O"]
)
plt.show()

> **Suggestions for exploration:**
> - Can you come up with a physical scenario where gas and dust temperature are decoupled?
> - Be creative with the CRIR and UV field. 
> - Try a density profile that *decreases* over time (e.g. an expanding shell). How does
>   the chemistry respond to a falling density?
> - Shocks! (be sure to add plenty of timesteps so the chemistry doesn't get stuck)
> - Timesteps don't need to be evenly spaced. Try a logarithmic time grid to capture early-time chemistry in more detail.

## Summary

In this notebook we have:

* Downloaded a NEATH SPH dataset and loaded its 16-column particle tracks into a pandas
  DataFrame.
* Used `uclchem.model.Postprocess` to run the UCLCHEM chemical network driven by real
  SPH particle histories, both for a single particle and for a small ensemble.
* Compared gas-phase methanol abundances across NEATH particles with a free-fall
  reference model, illustrating the changes introduced by realistic physical histories.
* Constructed a fully synthetic physical trajectory and explored how the chemistry responds
  to physical conditions you designed yourself.
